# NBA DFS Walk-Forward Backtest with Lineup Generation

This notebook runs the walk-forward backtesting pipeline with automatic lineup generation using pydfs-lineup-optimizer:
1. Load historical data for training
2. Build features using YAML-configured pipelines
3. Train XGBoost models (per-player or slate-level)
4. Generate predictions for each test slate
5. **Generate optimal lineups using pydfs-lineup-optimizer**
6. **Evaluate lineup performance against actuals**
7. Save predictions, models, and lineups to outputs directory

## Lineup Generation Features

- **Contest-specific strategies**: Cash games, GPPs, single-entry, multi-entry
- **Risk management**: Ceiling/floor projections, variance weighting
- **DraftKings constraints**: 8 players, $50K cap, position requirements
- **Exports**: CSV for upload, JSON for analysis
- **Performance tracking**: Projected vs actual lineup scores

## Results
Results are saved to `data/outputs/{timestamp}/` including:
- `predictions/` - Player projections
- `lineups/` - Generated lineups (CSV + JSON)
- `lineup_performance_report.csv` - Backtest lineup evaluation

## Setup

In [1]:
import sys
from pathlib import Path
import logging
import pandas as pd
from datetime import datetime, timedelta

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.optimization.backtest_lineup_integration import BacktestWithLineups
from src.data.loaders.historical_loader import HistoricalDataLoader

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print('Setup complete')

Setup complete


## System Resources

In [2]:
import psutil
import multiprocessing

cpu_count = multiprocessing.cpu_count()
ram_gb = psutil.virtual_memory().total / (1024**3)

print(f"CPU Cores: {cpu_count}")
print(f"RAM: {ram_gb:.1f} GB")
print(f"Recommended n_jobs: {cpu_count}")

if ram_gb < 12:
    print("WARNING: Low RAM detected. Consider reducing n_jobs or processing fewer players.")

CPU Cores: 32
RAM: 31.8 GB
Recommended n_jobs: 32


## Configuration

### Backtest Settings

In [3]:
# Paths
from src.config.paths import OUTPUTS_DIR, PROJECT_ROOT, DATA_DIR
OUTPUT_DIR = str(OUTPUTS_DIR)
DATA_DIR_PATH = str(DATA_DIR)

# Date ranges
TEST_START = 20250205
TEST_END = 20250210  # Small range for testing

# Model configuration
NUM_SEASONS = 2
FEATURE_CONFIG = 'default_features'
MODEL_TYPE = 'xgboost'
MIN_PLAYER_GAMES = 10
MIN_GAMES_FOR_BENCHMARK = 5
RECALIBRATE_DAYS = 7
SALARY_TIERS = [0, 4000, 6000, 8000, 15000]

# Execution settings
PER_PLAYER_MODELS = False
SAVE_MODELS = True
SAVE_PREDICTIONS = True
N_JOBS = 16  # Adjust based on your system

# Model parameters
MODEL_PARAMS = {
    'max_depth': 6,
    'learning_rate': 0.05,
    'n_estimators': 200,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'reg:squarederror',
    'random_state': 42
}

# Player filtering
FILTER_SALARY_MIN = 6000
FILTER_SALARY_MAX = None
FILTER_EXCLUDE_OUT = True
FILTER_EXCLUDE_DOUBTFUL = False
FILTER_EXCLUDE_QUESTIONABLE = False

print('Backtest Configuration:')
print(f'  Data Directory: {DATA_DIR_PATH}')
print(f'  Output Directory: {OUTPUT_DIR}')
print(f'  Testing Period: {TEST_START} to {TEST_END}')
print(f'  Model Type: {MODEL_TYPE}')
print(f'  Feature Config: {FEATURE_CONFIG}')
print(f'  Per-Player Models: {PER_PLAYER_MODELS}')
print(f'  Parallel Jobs: {N_JOBS}')
print(f'  Salary Tiers: {SALARY_TIERS}')

Backtest Configuration:
  Data Directory: c:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\data
  Output Directory: c:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\data\outputs
  Testing Period: 20250205 to 20250210
  Model Type: xgboost
  Feature Config: default_features
  Per-Player Models: False
  Parallel Jobs: 16
  Salary Tiers: [0, 4000, 6000, 8000, 15000]


### Lineup Generation Settings

In [4]:
# Lineup generation configuration
GENERATE_LINEUPS = True
CONTEST_CONFIG = 'cash_game.json'  # Options: cash_game.json, gpp_tournament.json, single_entry.json, multi_entry.json
NUM_LINEUPS = 10
TRACK_LINEUP_PERFORMANCE = True

print('='*80)
print('LINEUP GENERATION SETTINGS')
print('='*80)
print(f'Generate Lineups: {GENERATE_LINEUPS}')
if GENERATE_LINEUPS:
    print(f'Contest Config: {CONTEST_CONFIG}')
    print(f'Number of Lineups: {NUM_LINEUPS}')
    print(f'Track Performance: {TRACK_LINEUP_PERFORMANCE}')

# Display contest config details
if GENERATE_LINEUPS:
    import json
    config_path = repo_root / 'config' / 'contests' / CONTEST_CONFIG
    with open(config_path, 'r') as f:
        contest_cfg = json.load(f)
    
    print(f'\nContest Details:')
    print(f'  Name: {contest_cfg["name"]}')
    print(f'  Description: {contest_cfg["description"]}')
    print(f'  Strategy: {contest_cfg["optimization_settings"]["strategy"]}')
    print(f'  Risk Settings:')
    print(f'    Ceiling Weight: {contest_cfg["risk_settings"]["ceiling_weight"]}')
    print(f'    Floor Weight: {contest_cfg["risk_settings"]["floor_weight"]}')
    print(f'  Min Projected Points: {contest_cfg["player_settings"]["min_projected_points"]}')
    print(f'  Exclude Injured: {contest_cfg["player_settings"]["exclude_injured"]}')
    print(f'  Exclude Questionable: {contest_cfg["player_settings"]["exclude_questionable"]}')

LINEUP GENERATION SETTINGS
Generate Lineups: True
Contest Config: cash_game.json
Number of Lineups: 10
Track Performance: True

Contest Details:
  Name: Cash Game
  Description: 50/50s and Double-ups where top 40-50% win
  Strategy: floor
  Risk Settings:
    Ceiling Weight: 0.1
    Floor Weight: 0.9
  Min Projected Points: 20.0
  Exclude Injured: True
  Exclude Questionable: True


## Build Player Filters

In [5]:
from src.filters import ColumnFilter, InjuryFilter

player_filters = []

if FILTER_SALARY_MIN is not None:
    player_filters.append(ColumnFilter('salary', '>=', FILTER_SALARY_MIN))
    print(f'Added filter: salary >= {FILTER_SALARY_MIN}')

if FILTER_SALARY_MAX is not None:
    player_filters.append(ColumnFilter('salary', '<=', FILTER_SALARY_MAX))
    print(f'Added filter: salary <= {FILTER_SALARY_MAX}')

if FILTER_EXCLUDE_OUT or FILTER_EXCLUDE_DOUBTFUL or FILTER_EXCLUDE_QUESTIONABLE:
    injury_filter = InjuryFilter(
        exclude_out=FILTER_EXCLUDE_OUT,
        exclude_doubtful=FILTER_EXCLUDE_DOUBTFUL,
        exclude_questionable=FILTER_EXCLUDE_QUESTIONABLE
    )
    player_filters.append(injury_filter)
    excluded = []
    if FILTER_EXCLUDE_OUT:
        excluded.append('OUT')
    if FILTER_EXCLUDE_DOUBTFUL:
        excluded.append('DOUBTFUL')
    if FILTER_EXCLUDE_QUESTIONABLE:
        excluded.append('QUESTIONABLE')
    print(f'Added filter: exclude injury status {", ".join(excluded)}')

if player_filters:
    print(f'\nTotal filters: {len(player_filters)}')
else:
    print('No player filters configured')

Added filter: salary >= 6000
Added filter: exclude injury status OUT

Total filters: 2


## Run Backtest with Lineup Generation

Execute the walk-forward backtesting pipeline with automatic lineup generation for each slate.

In [6]:
# Calculate training period
test_end_dt = datetime.strptime(str(TEST_END), '%Y%m%d')
train_end = (test_end_dt - timedelta(days=1)).strftime('%Y%m%d')

if NUM_SEASONS == 1:
    train_start = HistoricalDataLoader.get_season_start_date(str(TEST_START))
else:
    train_start = HistoricalDataLoader.get_previous_season_start_date(str(TEST_START))

print(f'Calculated Training Period: {train_start} to {train_end}\n')

# Initialize backtest with lineup generation
backtest = BacktestWithLineups(
    train_start=train_start,
    train_end=train_end,
    test_start=TEST_START,
    test_end=TEST_END,
    model_type=MODEL_TYPE,
    model_params=MODEL_PARAMS,
    feature_config=FEATURE_CONFIG,
    output_dir=OUTPUT_DIR,
    data_dir=DATA_DIR_PATH,
    per_player_models=PER_PLAYER_MODELS,
    min_player_games=MIN_PLAYER_GAMES,
    min_games_for_benchmark=MIN_GAMES_FOR_BENCHMARK,
    recalibrate_days=RECALIBRATE_DAYS,
    num_seasons=NUM_SEASONS,
    salary_tiers=SALARY_TIERS,
    save_models=SAVE_MODELS,
    save_predictions=SAVE_PREDICTIONS,
    n_jobs=N_JOBS,
    player_filters=player_filters if player_filters else None,
    # Lineup generation parameters
    generate_lineups=GENERATE_LINEUPS,
    contest_config=CONTEST_CONFIG,
    num_lineups=NUM_LINEUPS,
    track_lineup_performance=TRACK_LINEUP_PERFORMANCE
)

print('\n' + '='*80)
print('RUNNING BACKTEST WITH LINEUP GENERATION')
print('='*80 + '\n')

# Run backtest
results = backtest.run_with_lineups()

if 'error' in results:
    print(f"\nERROR: {results['error']}")
else:
    print(f"\n{'='*80}")
    print('BACKTEST COMPLETED SUCCESSFULLY')
    print('='*80)

2025-10-22 11:11:10,622 - src.evaluation.backtest.walk_forward - WARNING - train_end (20250209) is after test_start (20250205). Training data will overlap with test window.
2025-10-22 11:11:10,636 - src.utils.feature_config - INFO - Loaded feature config: Default Feature Set
2025-10-22 11:11:10,637 - src.utils.feature_config - INFO - Added RollingStatsTransformer: windows=[3, 5, 10], stats=21, include_std=True
2025-10-22 11:11:10,637 - src.utils.feature_config - INFO - Added EWMATransformer: span=5, stats=21
2025-10-22 11:11:10,637 - src.utils.feature_config - INFO - Added TargetTransformer: target_col=fpts, shift_periods=-1
2025-10-22 11:11:10,638 - src.utils.feature_config - INFO - Added InjuryTransformer
2025-10-22 11:11:10,639 - src.evaluation.backtest.walk_forward - INFO - Initialized WalkForwardBacktest
2025-10-22 11:11:10,639 - src.evaluation.backtest.walk_forward - INFO - Data directory: c:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\data
2025-10-22 11:11:10,640

Calculated Training Period: 20231001 to 20250209


RUNNING BACKTEST WITH LINEUP GENERATION


Backtesting 6 slates from 20250205 to 20250210



Scanning slates:   0%|          | 0/6 [00:00<?, ?it/s]2025-10-22 11:11:10,758 - src.data.loaders.historical_loader - INFO - Loading slate data for 20250205
2025-10-22 11:11:10,842 - src.data.loaders.historical_loader - INFO - Loaded slate data: 380 salaries, 11 games
2025-10-22 11:11:10,845 - src.data.loaders.historical_loader - INFO - Loading slate data for 20250206
2025-10-22 11:11:10,888 - src.data.loaders.historical_loader - INFO - Loaded slate data: 208 salaries, 6 games
Scanning slates:  33%|███▎      | 2/6 [00:00<00:00, 15.09it/s]2025-10-22 11:11:10,891 - src.data.loaders.historical_loader - INFO - Loading slate data for 20250207
2025-10-22 11:11:10,940 - src.data.loaders.historical_loader - INFO - Loaded slate data: 242 salaries, 7 games
2025-10-22 11:11:10,943 - src.data.loaders.historical_loader - INFO - Loading slate data for 20250208
2025-10-22 11:11:11,022 - src.data.loaders.historical_loader - INFO - Loaded slate data: 379 salaries, 11 games
Scanning slates:  67%|██████▋ 


BACKTEST COMPLETED SUCCESSFULLY


## Results Summary

In [10]:
if 'error' not in results:
    print('='*80)
    print('BACKTEST RESULTS SUMMARY')
    print('='*80)
    print(f'\nNumber of Slates: {results.get("test_slates", 0)}')
    print(f'Total Players Evaluated: {results.get("total_players", 0)}')
    
    print(f'\nModel Performance:')
    if 'model_mean_mape' in results:
        print(f'  Mean MAPE: {results["model_mean_mape"]:.2f}%')
        print(f'  Mean RMSE: {results.get("model_mean_rmse", 0):.2f}')
        print(f'  Mean Correlation: {results.get("model_mean_correlation", 0):.3f}')
    
    if 'lineup_summary' in results:
        lineup_summary = results['lineup_summary']
        print(f'\nLINEUP GENERATION SUMMARY:')
        print(f'  Total Lineups Generated: {lineup_summary.get("total_lineups", 0)}')
        print(f'  Avg Projected vs Actual Correlation: {lineup_summary.get("avg_correlation", 0):.3f}')
        print(f'  Avg Error: {lineup_summary.get("avg_error_pct", 0):.1f}%')
    
    print(f'\n{'='*80}')
    print('OUTPUT LOCATION')
    print('='*80)
    print(f'\nResults saved to: {backtest.run_output_dir}')
    
    if GENERATE_LINEUPS:
        print(f'\nLineup files:')
        print(f'  CSV (DraftKings upload): {backtest.run_output_dir}/lineups/*_lineups.csv')
        print(f'  JSON (full details): {backtest.run_output_dir}/lineups/*_lineups.json')
        if TRACK_LINEUP_PERFORMANCE:
            print(f'  Performance report: {backtest.run_output_dir}/lineup_performance_report.csv')
else:
    print('Backtest failed. Check error message above.')

BACKTEST RESULTS SUMMARY

Number of Slates: 0
Total Players Evaluated: 0

Model Performance:
  Mean MAPE: 25.78%
  Mean RMSE: 10.89
  Mean Correlation: 0.601

OUTPUT LOCATION

Results saved to: c:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\data\outputs\20251022_111110

Lineup files:
  CSV (DraftKings upload): c:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\data\outputs\20251022_111110/lineups/*_lineups.csv
  JSON (full details): c:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\data\outputs\20251022_111110/lineups/*_lineups.json
  Performance report: c:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\data\outputs\20251022_111110/lineup_performance_report.csv


## Lineup Performance Analysis

In [11]:
if 'error' not in results and 'lineup_performance' in results:
    lineup_perf_df = results['lineup_performance']
    
    print('='*80)
    print('LINEUP PERFORMANCE DETAILS')
    print('='*80)
    print(lineup_perf_df.to_string(index=False))
    
    # Performance statistics
    print(f'\nPerformance Statistics:')
    print(f'  Avg Projected Score: {lineup_perf_df["avg_projected"].mean():.2f}')
    print(f'  Avg Actual Score: {lineup_perf_df["avg_actual"].mean():.2f}')
    print(f'  Best Actual Score: {lineup_perf_df["best_actual"].max():.2f}')
    print(f'  Worst Actual Score: {lineup_perf_df["worst_actual"].min():.2f}')
    print(f'  Avg Correlation: {lineup_perf_df["correlation"].mean():.3f}')
    print(f'  Avg Error %: {lineup_perf_df["avg_error_pct"].mean():.1f}%')
else:
    print('No lineup performance data available')

No lineup performance data available


## Sample Lineup Details

In [12]:
import json
from pathlib import Path

if 'error' not in results and GENERATE_LINEUPS:
    # Find lineup JSON files
    lineup_dir = Path(backtest.run_output_dir) / 'lineups'
    if lineup_dir.exists():
        json_files = list(lineup_dir.glob('*_lineups.json'))
        
        if json_files:
            # Load first file
            with open(json_files[0], 'r') as f:
                lineups_data = json.load(f)
            
            print('='*80)
            print(f'SAMPLE LINEUP DETAILS (from {json_files[0].name})')
            print('='*80)
            
            # Display first lineup
            if lineups_data:
                lineup = lineups_data[0]
                print(f'\nLineup #{lineup["lineup_num"]}')
                print(f'Total Salary: ${lineup["total_salary"]:,}')
                print(f'Projected Points: {lineup["projected_points"]:.2f}')
                print(f'Salary Remaining: ${lineup["salary_remaining"]:,}')
                print(f'Contest Type: {lineup["contest_type"]}')
                
                print(f'\n{"Position":<8} {"Player":<25} {"Team":<5} {"Salary":<10} {"Proj. Pts":<10}')
                print('-'*60)
                
                for player in lineup['players']:
                    pos = player.get('position', 'UTIL')
                    name = player.get('playerName', 'Unknown')
                    team = player.get('team', 'N/A')
                    salary = player.get('salary', 0)
                    proj_pts = player.get('projected_fpts', 0)
                    
                    print(f'{pos:<8} {name:<25} {team:<5} ${salary:<9,} {proj_pts:<10.2f}')
                
                print(f'\nTotal lineups in file: {len(lineups_data)}')
        else:
            print('No lineup JSON files found')
    else:
        print('Lineup directory not found')
else:
    print('Lineup generation was disabled or backtest failed')

Lineup directory not found


## Next Steps

### Using Generated Lineups

1. **Review Lineups**: Check CSV files in `{output_dir}/lineups/` directory
2. **Upload to DraftKings**: Use CSV files for direct contest entry
3. **Analyze Performance**: Review lineup_performance_report.csv for accuracy metrics
4. **Adjust Strategy**: Modify contest config files in `config/contests/` for different strategies

### Contest Strategy Guide

- **Cash Games** (`cash_game.json`): Conservative, floor-focused, single lineup
- **GPP Tournaments** (`gpp_tournament.json`): Aggressive, ceiling-focused, 20+ lineups
- **Single Entry** (`single_entry.json`): Balanced strategy, 1-3 lineups
- **Multi Entry** (`multi_entry.json`): Diversity-focused, 50+ lineups

### Further Analysis

Use `evaluate_backtest.ipynb` to:
- Generate detailed visualizations
- Analyze prediction accuracy by salary tier
- Compare model performance against benchmarks
- Generate PDF reports